In [16]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from PIL import Image
import os
import json

In [17]:
#Create the model class
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        #Same Padding = [(filter size - 1) / 2] (Same Padding--> input size = output size)
        self.cnn1 = nn.Conv2d(in_channels=3, out_channels=4, kernel_size=3,stride=1, padding=1)
        #The output size of each of the 4 feature maps is
        #[(input_size - filter_size + 2(padding) / stride) +1] --> [(64-3+2(1)/1)+1] = 64 (padding type is same)
        self.batchnorm1 = nn.BatchNorm2d(4)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)

        #After max pooling, the output of each feature map is now 64/2 =32
        self.cnn2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        #Output size of each of the 32 feature maps
        self.batchnorm2 = nn.BatchNorm2d(8)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)

        #After max pooling, the output of each feature map is 32/2 = 16
        #Flatten the feature maps. You have 8 feature maps, each of them is of size 16x16 --> 8*16*16 = 2048
        self.fc1 = nn.Linear(in_features=8*16*16, out_features=32)
        self.droput = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(in_features=32, out_features=10)

    def forward(self,x):
        out = self.cnn1(x)
        out = self.batchnorm1(out)
        out = self.relu(out)
        out = self.maxpool1(out)
        out = self.cnn2(out)
        out = self.batchnorm2(out)
        out = self.relu(out)
        out = self.maxpool2(out)

        #Now we have to flatten the output. This is where we apply the feed forward neural network as learned before!
        #It will take the shape (batch_size, 2048)
        out = out.view(x.size(0), -1)

        #Then we forward through our fully connected layer
        out = self.fc1(out)
        out = self.relu(out)
        #out = self.droput(out)
        out = self.fc2(out)
        return out

In [18]:
data_transforms = transforms.Compose([
        transforms.Resize(64),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

In [19]:
model = CNN()

USE_CUDA = torch.cuda.is_available()

if USE_CUDA:
  model.load_state_dict(torch.load('/content/saved_model.pth'))
  model.cuda()
else:
  model.load_state_dict(torch.load('/content/saved_model.pth', map_location=torch.device('cpu')))

model.eval()

CNN(
  (cnn1): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batchnorm1): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (cnn2): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batchnorm2): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (maxpool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2048, out_features=32, bias=True)
  (droput): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=32, out_features=10, bias=True)
)

In [20]:
folder_path = '/content/images'
result = {}
with torch.no_grad():  # Disable gradient calculation
    index = 0
    while True:
      image_path = os.path.join(folder_path, f'{index}.png')
      if os.path.exists(image_path):
        print(f'Processing {image_path}...', end='')

        image = Image.open(image_path)
        image = image.convert('RGB')
        image = data_transforms(image).unsqueeze(0)
        if USE_CUDA:
            image = image.cuda()

        outputs = model(image)
        _, predicted = torch.max(outputs, 1)  # Get the predicted class

        print(f'Predicted class: {predicted.item()}')  # Print the predicted class
        result[index] = predicted.item()
        index += 1
      else:
        break

Streaming output truncated to the last 5000 lines.
Processing /content/images/15740.png...Predicted class: 6
Processing /content/images/15741.png...Predicted class: 6
Processing /content/images/15742.png...Predicted class: 6
Processing /content/images/15743.png...Predicted class: 6
Processing /content/images/15744.png...Predicted class: 6
Processing /content/images/15745.png...Predicted class: 6
Processing /content/images/15746.png...Predicted class: 2
Processing /content/images/15747.png...Predicted class: 6
Processing /content/images/15748.png...Predicted class: 6
Processing /content/images/15749.png...Predicted class: 2
Processing /content/images/15750.png...Predicted class: 1
Processing /content/images/15751.png...Predicted class: 1
Processing /content/images/15752.png...Predicted class: 1
Processing /content/images/15753.png...Predicted class: 2
Processing /content/images/15754.png...Predicted class: 6
Processing /content/images/15755.png...Predicted class: 6
Processing /content/i

In [21]:
# Store all result to json file
result_json = json.dumps(result)

# Optionally, save to a file
with open('result.json', 'w') as json_file:
    json.dump(result, json_file)